# 한국어 음성을 텍스트로 — 무료 오픈소스 STT 실측 비교

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/korean_stt_comparison.ipynb)

무료로 쓸 수 있는 오픈소스 STT 네 가지를 **같은 음성으로** 돌려
정확도 · 처리 시간 · 라이선스를 나란히 비교합니다.

| 모델 | 특징 | 가중치 라이선스 |
|---|---|---|
| Whisper large-v3-turbo | 기준점. `transformers` 파이프라인 | MIT |
| faster-whisper | 같은 가중치를 CTranslate2 로 재구현 | MIT |
| WhisperX | faster-whisper + 강제 정렬 | BSD-4-Clause |
| CrisperWhisper 2.0 turbo | 말한 그대로(fillers 포함) 받아쓰기 | **비상업 연구용** |

> **CrisperWhisper 가중치는 상업적으로 쓸 수 없습니다.**
> 코드는 MIT 지만 가중치는 Nyra Health 비상업 연구용 라이선스입니다.
> 업무·제품에 넣으려면 별도 상업 라이선스를 받아야 합니다. 나머지 셋은 상업 이용이 가능합니다.

## 실행 순서

1. `런타임 → 런타임 유형 변경 → T4 GPU` 를 선택합니다.
2. 위에서부터 셀을 순서대로 실행합니다.
3. 설치 셀을 돌린 뒤 **반드시 `런타임 → 세션 다시 시작`** 을 누르고, 설치 셀 다음부터 이어서 실행합니다.
   numpy 가 교체되므로 재시작하지 않으면 그다음 셀이 `ImportError` 로 죽습니다.

영어 샘플은 노트북이 직접 내려받습니다. 한국어는 **직접 올린 음성**으로 비교합니다 —
개인 녹음은 저장소에 넣지 않기 때문에, 올리지 않으면 영어 샘플로만 돕니다.

## 1. 런타임 확인

GPU 없이도 돌지만 CPU 에서는 수 분씩 걸립니다.

In [ ]:
import subprocess, sys

print("Python", sys.version.split()[0])
try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=30).stdout.strip()
except (FileNotFoundError, subprocess.TimeoutExpired):
    gpu = ""
print("GPU:", gpu or "없음 — CPU 로 실행합니다 (느립니다)")

## 2. 설치

의존성을 한 번에 설치한 뒤 `ctranslate2` 를 CrisperWhisper 포크로 갈아 끼웁니다. **순서를 지켜야 합니다.**

- 한 번에 설치 — 나눠 깔면 pip 가 `ctranslate2` · `torch` 를 번갈아 올렸다 내렸다 하며 서로 깨뜨립니다.
- `whisperx>=3.8` — 열어 두면 pip 가 3.3 대를 고르는데, 그 버전은 지금 `torchaudio` 에서 import 되지 않습니다.
- `numba>=0.61` — Colab 에 깔린 numba 가 새 numpy 를 따라오지 못합니다.
- `ctranslate2` 교체 — CrisperWhisper 는 업스트림에 없는 API 를 쓰는 포크가 필요한데 WhisperX 가
  업스트림을 끌고 옵니다. 둘은 같은 디렉터리를 써서 **함께 지우고 포크만** 다시 깔아야 합니다.
  포크는 업스트림의 상위 집합이라 faster-whisper · WhisperX 도 그대로 돕니다.

pip 가 `google-colab` · gradio · pytensor · cudf 충돌을 경고하지만, 이 노트북이 쓰는
`google.colab` 업로드 위젯은 정상 동작합니다.

In [ ]:
%pip install -q "crisperwhisper[ct2]" faster-whisper transformers accelerate \
    "whisperx>=3.8" "numba>=0.61"

# CrisperWhisper 포크로 교체한다. 순서가 중요하다 — 위 설치 뒤에 와야 한다.
# 둘을 함께 지운다. 같은 `ctranslate2/` 디렉터리를 공유해서, 업스트림만 지우면
# 포크의 파일까지 날아가고 메타데이터만 남아 재설치가 "이미 설치됨" 으로 건너뛴다.
%pip uninstall -q -y ctranslate2 ctranslate2-crisperwhisper
%pip install -q ctranslate2-crisperwhisper

import importlib.metadata as md

for pkg in ("numpy", "numba", "ctranslate2", "ctranslate2-crisperwhisper", "transformers",
            "faster-whisper", "whisperx", "crisperwhisper", "torch"):
    try:
        print(f"{pkg:16} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:16} 없음")

print("\n설치 완료 — '런타임 → 세션 다시 시작' 을 누르고 다음 셀부터 이어서 실행하세요.")
print("numpy 가 교체돼서, 재시작하지 않으면 이전 버전이 메모리에 남아 import 가 깨집니다.")

## 3. 샘플 오디오 내려받기 (영어)

**AMI Meeting Corpus** ES2004a 회의 녹음입니다. 원본은 17분·32MB 라 26초만 잘라 씁니다.

자르는 구간(`116.5초`부터)은 **한 사람만 말하는 대목**으로 골랐습니다. 회의 앞부분은 여러 명이
겹쳐 말해서, 받아쓴 결과를 정답과 맞춰 보면 누가 틀린 게 아니라 어순이 엉켜 오인식으로 잡힙니다.

- 출처: <https://groups.inf.ed.ac.uk/ami/corpus/>
- 라이선스: **CC BY 4.0** — 재배포·상업 이용이 허용됩니다.

In [ ]:
# AMI Meeting Corpus ES2004a (Mix-Headset)
# 출처: https://groups.inf.ed.ac.uk/ami/corpus/ · 라이선스: CC BY 4.0 (재배포 허용)
import os, subprocess, urllib.request

AMI_URL = ("https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/"
           "ES2004a/audio/ES2004a.Mix-Headset.wav")
SRC, CLIP_EN = "ami_es2004a.wav", "clip_en.wav"   # 상대 경로 — 노트북 폴더 기준

if not os.path.exists(SRC):
    print("내려받는 중… (약 32MB)")
    urllib.request.urlretrieve(AMI_URL, SRC)

# 116.5초부터 26초. 화자 한 명만 말하고 앞뒤로도 겹치지 않는 구간이다.
# 겹쳐 말하는 대목을 쓰면 어순이 엉켜 오인식이 부풀려진다. 16kHz 모노로 통일한다.
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", SRC,
                "-ss", "116.5", "-t", "26", "-ar", "16000", "-ac", "1", CLIP_EN], check=True)
print(f"{CLIP_EN} — {os.path.getsize(CLIP_EN) / 1024:.0f}KB (26초, 16kHz 모노)")

## 4. 한국어 음성 올리기 (선택)

**개인 녹음은 저장소에 들어 있지 않습니다.** 한국어로 비교하려면 직접 올리세요 —
30초 안팎, 어떤 형식이든 됩니다(`m4a` · `mp3` · `wav`). 올린 파일은 Colab 세션에만 남고
세션이 끝나면 사라집니다.

아래 `REFERENCE_KO` 에 **실제로 말한 내용**을 적으면 모델별 오인식 개수를 세어 줍니다.
비우면 그 칸은 `-` 로 나옵니다.

**올리지 않아도 오인식 칸은 채워집니다.** 영어 샘플에는 AMI 공식 수동 전사를 정답으로 넣어 뒀습니다.

In [ ]:
import os, subprocess

CLIP_KO = "clip_ko.wav"
try:
    from google.colab import files
    print("한국어 음성 파일을 고르세요. 건너뛰려면 '취소' 를 누르세요.")
    up = files.upload()
except ImportError:                      # Colab 이 아니면 같은 폴더의 파일을 찾는다
    up = {f: None for f in os.listdir(".") if f.startswith("my_audio.")}

# 올린 음성에서 실제로 말한 내용. 오인식 개수 계산에만 씁니다 — 비워도 됩니다.
REFERENCE_KO = ""

# 영어 샘플의 정답. AMI 공식 수동 전사(ES2004a 화자 B)의 116.5~142.5초 구간입니다.
# 출처: ami_public_manual_1.6.2.zip 의 words/ES2004a.B.words.xml · CC BY 4.0
REFERENCE_EN = """
Now, we're developing a remote control which you probably already know. Um, we want it
to be original, something that's uh people haven't thought of, that's not out in the
shops, um, trendy, appealing to a wide market, but you know, not a hunk of metal, and
user-friendly, grannies to kids, maybe even pooches should be able to use it
"""

if up:
    src = next(iter(up))
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src,
                    "-ar", "16000", "-ac", "1", CLIP_KO], check=True)
    AUDIO, LANG, REFERENCE = CLIP_KO, "ko", REFERENCE_KO
else:
    AUDIO, LANG, REFERENCE = CLIP_EN, "en", REFERENCE_EN

REFERENCE = " ".join(REFERENCE.split())
print(f"비교에 쓸 음성: {AUDIO} ({LANG}) · 정답 {'있음' if REFERENCE else '없음'}")

## 5. 공통 도구

네 모델을 같은 잣대로 재기 위한 것들입니다.

- **처리 시간** — 모델 로딩은 빼고 받아쓰기에 걸린 시간만 잽니다.
- **오인식 개수** — `REFERENCE` 와 단어 단위로 맞춰 보고 어긋난 단어를 셉니다.
- **필러** — `[UM]` 같은 머뭇거림을 몇 개나 남겼는지입니다. 많고 적음이 곧 좋고 나쁨은 아닙니다.
- **U+FFFD** — 디코딩이 깨질 때 나오는 대체 문자(`�`)의 개수입니다.

오인식을 셀 때 **필러는 양쪽에서 빼고** 셉니다. CrisperWhisper 처럼 말버릇을 남기는 모델과
지우는 모델을 같은 잣대로 놓기 위해서입니다. 표기가 갈리는 말(`세 군데` ↔ `3군데`)은
`SYNONYMS` 에 쌍으로 넣으면 같은 값으로 봅니다 — 자기 음성에 맞게 채우세요.

In [ ]:
import difflib, gc, re, time, traceback
from unicodedata import east_asian_width

RESULTS = {}

def free():
    """다음 모델을 올리기 전에 GPU 메모리를 비운다. T4 16GB 로는 네 개를 동시에 못 올린다."""
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass

def count_fffd(text):
    return text.count("\ufffd")

# 오인식을 세기 전에 양쪽을 같은 잣대로 만든다.
# 머뭇거림과 주석 태그. 대괄호는 토큰으로 쪼갤 때 떨어지므로 `[UH]` 는 `uh` 로 들어온다.
FILLERS = {"어", "음", "uh", "um", "laugh", "laughter", "vocalsound", "disfmarker"}
SYNONYMS = []                        # 같은 값으로 볼 표기 쌍 — 예: [("세 군데", "3군데")]

def tokens(text):
    for a, b in SYNONYMS:
        text = text.replace(a, b)
    return re.findall(r"[0-9A-Za-z가-힣]+", text.lower())

def count_fillers(text):
    return sum(1 for t in tokens(text) if t in FILLERS)

def word_errors(ref, hyp):
    """ref 대비 어긋난 단어 수. ref 가 비어 있으면 None.

    필러는 양쪽에서 빼고 셉니다 — 안 그러면 말버릇을 버리는 모델이 일방적으로 불리해집니다.
    필러를 남겼는지는 오인식이 아니라 별도 열(`필러`)로 봅니다.
    """
    if not ref.strip():
        return None
    drop = lambda ts: [t for t in ts if t not in FILLERS]
    a, b = drop(tokens(ref)), drop(tokens(hyp))
    sm = difflib.SequenceMatcher(a=a, b=b, autojunk=False)
    return sum(max(i2 - i1, j2 - j1)
               for tag, i1, i2, j1, j2 in sm.get_opcodes() if tag != "equal")

def measure(name, license_, fn):
    """받아쓰기 한 번을 재고 RESULTS 에 담는다. 하나가 실패해도 나머지는 계속 돈다."""
    t = time.perf_counter()
    try:
        text = fn().strip()
        RESULTS[name] = {"text": text, "sec": time.perf_counter() - t, "license": license_}
        print(f"{name} — {RESULTS[name]['sec']:.1f}초 · {len(text)}자 · U+FFFD {count_fffd(text)}개")
    except Exception:
        RESULTS[name] = {"text": "", "sec": None, "license": license_, "failed": True}
        traceback.print_exc()
        print(f"{name} — 실패 (표에는 '-' 로 남습니다)")

def dwidth(s):
    """터미널 표시 폭. 한글·한자는 두 칸을 차지한다."""
    return sum(2 if east_asian_width(c) in "WF" else 1 for c in s)

def pad(s, n):
    return s + " " * max(0, n - dwidth(s))

def wrap_wide(text, width):
    """표시 폭 기준으로 접는다. textwrap 은 글자 수로 세어 한글에서 두 배로 넘친다."""
    lines, line = [], ""
    for word in text.split():
        candidate = f"{line} {word}" if line else word
        if line and dwidth(candidate) > width:
            lines.append(line)
            line = word
        else:
            line = candidate
        while dwidth(line) > width:          # 한 단어가 통째로 넘칠 때
            cut = len(line)
            while cut > 1 and dwidth(line[:cut]) > width:
                cut -= 1
            lines.append(line[:cut])
            line = line[cut:]
    if line:
        lines.append(line)
    return lines

print("준비 완료")

## 6. Whisper large-v3-turbo

OpenAI 원본 가중치를 `transformers` 파이프라인으로 돌립니다. 나머지 셋의 기준점입니다.

In [ ]:
import torch
from transformers import pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

asr = pipeline("automatic-speech-recognition", "openai/whisper-large-v3-turbo",
               device=device, dtype=dtype)

measure("Whisper large-v3-turbo", "MIT",
        lambda: asr(AUDIO, generate_kwargs={"language": LANG}, return_timestamps=True)["text"])

del asr
free()

## 7. faster-whisper

같은 가중치를 CTranslate2 로 재구현한 것입니다. 결과는 거의 같고 속도·메모리가 다릅니다.

In [ ]:
from faster_whisper import WhisperModel

fw = WhisperModel("large-v3-turbo", device=device,
                  compute_type="float16" if device == "cuda" else "int8")

def run_fw():
    segments, _ = fw.transcribe(AUDIO, language=LANG)
    return " ".join(s.text.strip() for s in segments)   # 제너레이터라 여기서 실제로 계산된다

measure("faster-whisper", "MIT", run_fw)

del fw
free()

## 8. WhisperX

faster-whisper 로 받아쓴 뒤 음소 정렬 모델로 단어 경계를 다시 맞춥니다.
여기서는 받아쓰기 시간만 재고 정렬 단계는 빼서 다른 셋과 조건을 맞춥니다.

In [ ]:
import whisperx

wx = whisperx.load_model("large-v3-turbo", device,
                         compute_type="float16" if device == "cuda" else "int8",
                         language=LANG)

def run_wx():
    audio = whisperx.load_audio(AUDIO)
    out = wx.transcribe(audio, batch_size=8)
    return " ".join(s["text"].strip() for s in out["segments"])

measure("WhisperX", "BSD-4-Clause", run_wx)

del wx
free()

## 9. CrisperWhisper 2.0 turbo

말한 그대로 받아씁니다 — `[UM]` · `[UH]` 같은 머뭇거림과 더듬은 말까지 남깁니다.
회의록·인터뷰처럼 **말버릇이 정보인** 경우에 씁니다.

> **가중치는 비상업 연구용입니다.** 코드는 MIT 지만 가중치는 Nyra Health 비상업 연구용
> 라이선스라 상업적으로 쓸 수 없습니다. 제품에 넣으려면 상업 라이선스를 따로 받아야 합니다.

`mode="verbatim"` 이 말한 그대로, `mode="intended"` 가 다듬은 결과입니다. 둘 다 재 봅니다.

In [ ]:
from crisperwhisper import CrisperWhisperModel

# backend 를 비우면 GPU 에서 ct2, 그 밖에는 transformers 를 고릅니다.
# Mac 처럼 float16 을 못 쓰는 환경에서는 backend="transformers", compute_type="float32" 를 넘기세요.
cw = CrisperWhisperModel("nyralabs/CrisperWhisper2.0_turbo", device="auto")

for mode, label in (("verbatim", "CrisperWhisper (verbatim)"),
                    ("intended", "CrisperWhisper (intended)")):
    measure(label, "비상업 연구용",
            lambda m=mode: cw.transcribe(AUDIO, language=LANG, mode=m).text)

del cw
free()

## 10. 비교 표

처리 시간 · 오인식 · 필러 · 라이선스를 한 표에 놓습니다. `REFERENCE` 를 비웠으면 오인식 칸은 `-` 입니다.

시간은 **각 라이브러리의 기본 설정** 기준입니다. 같은 조건이 아닙니다 —
faster-whisper 는 beam 5 로 탐색하고 CrisperWhisper 는 greedy 로 한 번에 갑니다.
탐색 폭을 맞추고 싶으면 `fw.transcribe(..., beam_size=1)` 로 두고 다시 재 보세요.

In [ ]:
W_NAME, W_SEC, W_ERR, W_FIL = 27, 9, 7, 5

print(pad("모델", W_NAME) + pad("시간(초)", W_SEC)
      + pad("오인식", W_ERR) + pad("필러", W_FIL) + "라이선스")
print("-" * 64)
for name, r in RESULTS.items():
    sec = f"{r['sec']:.1f}" if r["sec"] is not None else "-"
    err = None if r.get("failed") else word_errors(REFERENCE, r["text"])
    fil = "-" if r.get("failed") else str(count_fillers(r["text"]))
    print(pad(name, W_NAME) + pad(sec, W_SEC)
          + pad("-" if err is None else str(err), W_ERR) + pad(fil, W_FIL) + r["license"])
print("-" * 64)
print(f"음성: {AUDIO} ({LANG}) · 장치: {device}")

## 11. 터미널 이미지용 텍스트

글에 넣을 캡처용입니다. 한 줄이 **64칸을 넘지 않게** 접습니다 —
한글은 한 글자가 두 칸이라 글자 수가 아니라 표시 폭으로 세야 합니다.

In [ ]:
WIDTH = 64

print("=" * WIDTH)
print(f" STT 비교 — {AUDIO} ({LANG})")
print("=" * WIDTH)
for i, (name, r) in enumerate(RESULTS.items(), 1):
    sec = f"{r['sec']:.1f}s" if r["sec"] is not None else "실패"
    head = f"[{i}] {name}"
    print()
    print(pad(head, WIDTH - dwidth(sec)) + sec)
    print("-" * WIDTH)
    for line in wrap_wide(r["text"] or "(결과 없음)", WIDTH):
        print(line)

## 12. 한글이 깨지는 이유

CrisperWhisper 의 한국어 결과에 `�` 가 섞여 나오면 **모델이 못 알아들은 게 아닙니다.**
단어 단위 타임스탬프를 만들려고 토큰을 하나씩 따로 디코딩하는데, 한글 한 글자는 UTF-8 로 3바이트라
토큰 경계에서 잘립니다. 영어는 한 글자가 1바이트라 멀쩡합니다.

아래 셀이 그 현상만 떼어내 보여줍니다 — 모델 없이 토크나이저만으로 재현됩니다.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("nyralabs/CrisperWhisper2.0_turbo")
sample = "클라우드 네이버 요금 비교"

ids = tok.encode(sample, add_special_tokens=False)
whole = tok.decode(ids)                             # 한 번에 디코딩 — 정상
per_token = "".join(tok.decode([i]) for i in ids)   # 토큰마다 따로 — 깨진다

print("원문      :", sample)
print("통째 디코딩:", whole)
print("토큰별    :", per_token)
print()
print("토큰 수        :", len(ids))
print("U+FFFD 개수    :", count_fffd(per_token))
print("왕복 일치 여부 :", whole == sample)

## 정리

- **상업적으로 쓸 것** — Whisper · faster-whisper(MIT), WhisperX(BSD). 속도가 중요하면 faster-whisper.
- **말버릇까지 필요할 것** — CrisperWhisper. 단, **가중치가 비상업 연구용**이라 제품에는 못 넣습니다.
- 한국어 `�` 는 모델 성능이 아니라 디코딩 방식 문제입니다. 통째로 디코딩하면 멀쩡합니다.

숫자는 GPU · 오디오 길이 · 발화 속도에 따라 달라집니다. **직접 올린 음성으로 다시 재 보세요.**